In [3]:
import random
import numpy as np

# A. Set up the environment
# 0 = empty, 1 = AI (X), -1 = opponent (O)
Q = {}
alpha, gamma, epsilon = 0.8, 0.9, 1.0


# B. Define the Tic-Tac-Toe game
def moves(board):
    return [i for i in range(9) if board[i] == 0]


def winner(board):
    lines = [(0,1,2), (3,4,5), (6,7,8),
             (0,3,6), (1,4,7), (2,5,8),
             (0,4,8), (2,4,6)]

    for a, b, c in lines:
        if board[a] != 0 and board[a] == board[b] == board[c]:
            return board[a]
    return 0


def show_board(board):
    s = [str(i + 1) if v == 0 else ("X" if v == 1 else "O")
         for i, v in enumerate(board)]
    for i in range(0, 9, 3):
        print(" | ".join(s[i:i+3]))
        if i < 6:
            print("---------")


# C. Build the Q-Learning model
def values(board):
    return Q.setdefault(tuple(board), np.zeros(9))


def choose_action(board, eps):
    legal = moves(board)

    if random.random() < eps:       # Explore
        return random.choice(legal)

    q = values(board)               # Exploit learned knowledge
    best = max(q[a] for a in legal)
    return random.choice([a for a in legal if q[a] == best])


# D. Train the model against a random opponent
for episode in range(15000):
    board = [0] * 9
    done = False

    while not done:
        state = tuple(board)
        action = choose_action(board, epsilon)
        board[action] = 1

        result = winner(board)

        if result == 1:
            reward, done = 1, True
        elif not moves(board):
            reward, done = 0, True
        else:
            # Opponent plays randomly
            board[random.choice(moves(board))] = -1
            result = winner(board)

            if result == -1:
                reward, done = -1, True
            elif not moves(board):
                reward, done = 0, True
            else:
                reward = 0

        # Update Q-value
        next_state = tuple(board)

        if done:
            target = reward
        else:
            target = reward + gamma * max(
                values(next_state)[a] for a in moves(board)
            )

        values(state)[action] += alpha * (
            target - values(state)[action]
        )

    epsilon = max(0.05, epsilon * 0.9997)

print("Training completed!")


# E. Test the trained AI against a human
board = [0] * 9
print("\nYou are O. AI is X. Positions are numbered 1 to 9.")

while True:
    # AI's turn
    action = choose_action(board, 0)
    board[action] = 1
    print("\nAI's move:")
    show_board(board)

    if winner(board) == 1:
        print("AI wins!")
        break
    if not moves(board):
        print("Draw!")
        break

    # Human's turn
    while True:
        try:
            pos = int(input("Enter your move (1-9): ")) - 1
            if pos in moves(board):
                break
            print("Invalid position. Try again.")
        except ValueError:
            print("Enter a number from 1 to 9.")

    board[pos] = -1

    if winner(board) == -1:
        show_board(board)
        print("You win!")
        break
    if not moves(board):
        show_board(board)
        print("Draw!")
        break

Training completed!

You are O. AI is X. Positions are numbered 1 to 9.

AI's move:
1 | 2 | 3
---------
4 | X | 6
---------
7 | 8 | 9


Enter your move (1-9):  2



AI's move:
1 | O | 3
---------
X | X | 6
---------
7 | 8 | 9


Enter your move (1-9):  6



AI's move:
1 | O | 3
---------
X | X | O
---------
X | 8 | 9


Enter your move (1-9):  1



AI's move:
O | O | X
---------
X | X | O
---------
X | 8 | 9
AI wins!
